In [1]:
print("""
@File         : dateoffsets.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-08 22:25:34
@Email        : cuixuanstephen@gmail.com
@Description  : 日期偏移量
""")


@File         : dateoffsets.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-08 22:25:34
@Email        : cuixuanstephen@gmail.com
@Description  : 日期偏移量



In [2]:
import pandas as pd

`pd.Timedelta` 用来将日期时间偏移有限的时间，例如 10 秒或 5 天。但是，`pd.Timedelta` 不能用于将日期或日期时间偏移一个月，因为一个月并不总是代表相同的时间长度。只需使用 `pd.DateOffset` 对象即可根据日历调整日期。

In [3]:
ser = pd.Series([
    "2024-01-01",
    "2024-01-02",
    "2024-01-03",
], dtype="datetime64[ns]")
ser

0   2024-01-01
1   2024-01-02
2   2024-01-03
dtype: datetime64[ns]

将这些日期移动一个月通常意味着保留同一天，但将日期放在二月而不是一月。使用 `pd.DateOffset`，可以向 `months=`
传递一个参数，该参数指示要将日期移动的月份数：

In [4]:
ser + pd.DateOffset(months=1)

0   2024-02-01
1   2024-02-02
2   2024-02-03
dtype: datetime64[ns]

In [5]:
ser + pd.DateOffset(months=2)

0   2024-03-01
1   2024-03-02
2   2024-03-03
dtype: datetime64[ns]

对于不存在的日期（例如，尝试将 1 月 30 日移至 2 月 30 日），`pd.DateOffset` 将尝试匹配目标月份内存在的最接近的日期：

In [6]:
pd.Series([
    "2024-01-29",
    "2024-01-30",
    "2024-01-31",
], dtype="datetime64[ns]") + pd.DateOffset(months=1)

0   2024-02-29
1   2024-02-29
2   2024-02-29
dtype: datetime64[ns]

还可以使用 `months=` 的负数参数来倒退日历：

In [7]:
ser + pd.DateOffset(months=-1)

0   2023-12-01
1   2023-12-02
2   2023-12-03
dtype: datetime64[ns]

In [8]:
ser + pd.DateOffset(months=1, days=2, hours=3, minutes=4, seconds=5)

0   2024-02-03 03:04:05
1   2024-02-04 03:04:05
2   2024-02-05 03:04:05
dtype: datetime64[ns]

除了 `pd.DateOffset` 类之外，pandas 还提供了将日期移至某个时期的开始或结束的功能，比如本月初、本月末、下月初。

In [9]:
ser + pd.offsets.MonthEnd()

0   2024-01-31
1   2024-01-31
2   2024-01-31
dtype: datetime64[ns]

In [10]:
ser + pd.offsets.MonthBegin()
# 下月初

0   2024-02-01
1   2024-02-01
2   2024-02-01
dtype: datetime64[ns]

In [11]:
ser + pd.offsets.SemiMonthBegin()

0   2024-01-15
1   2024-01-15
2   2024-01-15
dtype: datetime64[ns]

In [12]:
ser + pd.offsets.SemiMonthEnd()

0   2024-01-15
1   2024-01-15
2   2024-01-15
dtype: datetime64[ns]

[issues #8435](https://github.com/pandas-dev/pandas/issues/8435) 3 月作为季度月的第一个月

In [13]:
ser + pd.offsets.QuarterBegin() # ???

0   2024-03-01
1   2024-03-01
2   2024-03-01
dtype: datetime64[ns]

In [14]:
ser + pd.offsets.YearEnd()

0   2024-12-31
1   2024-12-31
2   2024-12-31
dtype: datetime64[ns]

`pd.DateOffset` 默认情况下是根据公历计算的，但它的不同子类可以提供更多定制功能。最常用的子类之一是 `pd.offsets.BusinessDay`，默认情况下，它在偏移日期时仅计算周一至周五的标准“工作日”。要了解其工作原理，让我们考虑一下 `ser` 中每个日期所在的星期几：

In [15]:
ser.dt.day_name()

0       Monday
1      Tuesday
2    Wednesday
dtype: object

In [17]:
bd_ser = ser + pd.offsets.BusinessDay(n=3)
bd_ser

0   2024-01-04
1   2024-01-05
2   2024-01-08
dtype: datetime64[ns]

In [18]:
bd_ser.dt.day_name()

0    Thursday
1      Friday
2      Monday
dtype: object

If you work with a business that has different business days from Monday to Friday, you could use the `pd.offsets.CustomBusinessDay` to set up your own rules for how offsetting should work. The argument to `weekmask=` will dictate the days of the week that are considered business days:

In [19]:
ser + pd.offsets.CustomBusinessDay(n=3, weekmask='Mon Tue Wed Thu')

C:\Users\JPL-JUNO\AppData\Local\Temp\ipykernel_4376\1283659793.py:1: PerformanceWarning: Non-vectorized DateOffset being applied to Series or DatetimeIndex.
  ser + pd.offsets.CustomBusinessDay(n=3, weekmask='Mon Tue Wed Thu')


0   2024-01-04
1   2024-01-08
2   2024-01-09
dtype: datetime64[ns]

甚至可以添加 `holidays=` 参数来说明公司可能停业的日子：

In [20]:
ser + pd.offsets.CustomBusinessDay(n=3, weekmask='Mon Tue Wed Thu', holidays=['2024-01-04'])

C:\Users\JPL-JUNO\AppData\Local\Temp\ipykernel_4376\3800314245.py:1: PerformanceWarning: Non-vectorized DateOffset being applied to Series or DatetimeIndex.
  ser + pd.offsets.CustomBusinessDay(n=3, weekmask='Mon Tue Wed Thu', holidays=['2024-01-04'])


0   2024-01-08
1   2024-01-09
2   2024-01-10
dtype: datetime64[ns]

我们已经看到了 `pd.offsets.MonthEnd` 和 `pd.offsets.MonthBegin` 类，它们分别可帮助将日期移至月初或月底。当尝试将日期移向工作月的开始或结束时，可以使用类似的类：

In [21]:
ser + pd.offsets.BusinessMonthEnd()

0   2024-01-31
1   2024-01-31
2   2024-01-31
dtype: datetime64[ns]